# Notebook 02: Phân chia Tập dữ liệu và Đánh giá Baseline (Data Splitting & Baseline Models)
* **Mục tiêu:** Tách riêng một nhóm quốc gia để làm tập kiểm thử (đóng băng dữ liệu), thiết lập ngưỡng phân loại mức độ trầm cảm, và chạy các mô hình học máy cơ sở để làm hệ quy chiếu so sánh hiệu năng với Hệ mờ ở giai đoạn sau.

### 1. Khởi tạo và Nạp dữ liệu
Import các thư viện `scikit-learn` cần thiết và nạp lại tập dữ liệu 204 quốc gia đã qua bước làm sạch từ Notebook 01. Quá trình này giúp Notebook 02 hoạt động độc lập khi chạy bằng lệnh Run All.

In [1]:
import os
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, f1_score

# Tự động nhận diện đường dẫn trên Kaggle hoặc VS Code (GitHub repo)
KAGGLE_PATH = "/kaggle/input/datasets/imtkaggleteam/mental-health/1- mental-illnesses-prevalence.csv"
LOCAL_PATH = "../data/1- mental-illnesses-prevalence.csv"
OUT_DIR = "../outputs" if os.path.exists("../outputs") else "."

df = pd.read_csv(KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH)
keys = {"depress":"depression", "schizo":"schizophrenia", "anxiety":"anxiety",
        "bipolar":"bipolar", "eating":"eating"}
df = df.rename(columns={c:v for c in df.columns for k,v in keys.items() if k in c.lower()})
df = df[df["Code"].notna() & ~df["Code"].astype(str).str.startswith("OWID")].copy()
feats = ["anxiety", "bipolar", "schizophrenia", "eating"]

print("Đã nạp dữ liệu:", len(df), "dòng |", df["Entity"].nunique(), "quốc gia")

Đã nạp dữ liệu: 6120 dòng | 204 quốc gia


### 2. Phân chia Train/Test và Thiết lập Ngưỡng phân loại
* **Chiến lược tách cụm:** Sử dụng `GroupShuffleSplit` để tách ngẫu nhiên nhưng giữ trọn vẹn 30 năm của cùng một quốc gia. Giữ lại 20% số quốc gia làm tập Test, danh sách này được xuất ra file CSV để đóng băng.
* **Ngưỡng phân loại:** Tính toán phân vị (1/3 và 2/3) của biến `depression` để chia 3 nhãn: Thấp (0), Vừa (1), Cao (2). **Lưu ý:** Ngưỡng này bắt buộc chỉ được tính toán trên tập Train để phòng tránh rò rỉ dữ liệu (data leakage).

In [2]:
# ---- 02_split: giữ lại 20% quốc gia làm tập test ----
import json
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(df, groups=df["Entity"]))
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
pd.Series(sorted(train.Entity.unique())).to_csv(f"{OUT_DIR}/countries_train.csv", index=False)
pd.Series(sorted(test.Entity.unique())).to_csv(f"{OUT_DIR}/countries_test.csv", index=False)

q1, q2 = train["depression"].quantile([1/3, 2/3])  # ngưỡng CHỈ tính trên train
json.dump({"q1": float(q1), "q2": float(q2)}, open(f"{OUT_DIR}/thresholds.json", "w"))
lab = lambda s: pd.cut(s, [-np.inf, q1, q2, np.inf], labels=[0,1,2]).astype(int)
train["y"], test["y"] = lab(train["depression"]), lab(test["depression"])
print(f"Số nước: {df['Entity'].nunique()} -> {train['Entity'].nunique()} train + {test['Entity'].nunique()} test")
print("ngưỡng", round(q1,3), round(q2,3))

Số nước: 204 -> 163 train + 41 test
ngưỡng 3.184 4.162


In [3]:
# Kiểm tra tương quan trên tập train và tái lập giả thuyết 99% phương sai
cols = feats + ["depression"]

print("--- 1. Tương quan trên toàn bộ dòng train (4.890 dòng) ---")
display(train[cols].corr().round(2))

print("--- 2. Tương quan theo trung bình quốc gia (163 nước) ---")
display(train.groupby("Entity")[cols].mean().corr().round(2))

print("--- 3. Kiểm chứng tỷ lệ phương sai trầm cảm giữa các quốc gia (toàn bộ 204 nước) ---")
var_between = df.groupby("Entity")["depression"].transform("mean").var(ddof=0)
var_total = df["depression"].var(ddof=0)
print(f"Tỷ lệ phương sai giữa các nước: {var_between / var_total:.4f}")

--- 1. Tương quan trên toàn bộ dòng train (4.890 dòng) ---


,anxiety,bipolar,schizophrenia,eating,depression
anxiety,1.00,0.59,0.26,0.59,0.13
bipolar,0.59,1.00,0.05,0.67,0.14
schizophrenia,0.26,0.05,1.00,0.43,-0.49
eating,0.59,0.67,0.43,1.00,-0.07
depression,0.13,0.14,-0.49,-0.07,1.00


--- 2. Tương quan theo trung bình quốc gia (163 nước) ---


,anxiety,bipolar,schizophrenia,eating,depression
anxiety,1.00,0.59,0.26,0.59,0.13
bipolar,0.59,1.00,0.05,0.68,0.14
schizophrenia,0.26,0.05,1.00,0.43,-0.50
eating,0.59,0.68,0.43,1.00,-0.07
depression,0.13,0.14,-0.50,-0.07,1.00


--- 3. Kiểm chứng tỷ lệ phương sai trầm cảm giữa các quốc gia (toàn bộ 204 nước) ---
Tỷ lệ phương sai giữa các nước: 0.9783


### 3. Đánh giá các Mô hình Đối chứng (Baseline Models)
Thực hiện huấn luyện và đánh giá trên tập Train thông qua kỹ thuật Cross-Validation (`GroupKFold` = 5). Khảo sát 3 mô hình:
1. **Majority:** Hệ cơ sở đoán ngây thơ theo nhãn phổ biến nhất.
2. **Logistic Regression:** Hệ tuyến tính chuẩn hóa.
3. **Decision Tree:** Cây quyết định (độ sâu tối đa = 3). 
Kết quả (Accuracy, Macro F1) được lưu vào file `baseline_results.csv` để làm mốc đối chuẩn cho Hệ suy diễn mờ ở Tuần 2.

In [4]:
# ---- 03_baseline: Đánh giá Cross-Validation trên Train (báo mean ± std) ----
from sklearn.model_selection import cross_validate

cv = GroupKFold(5)

# 1. Đóng băng danh sách fold theo từng quốc gia để hệ mờ dùng lại
fold = np.zeros(len(train), int)
for k, (_, va) in enumerate(cv.split(train, groups=train["Entity"])):
    fold[va] = k
train.assign(fold=fold)[["Entity", "fold"]].drop_duplicates().to_csv(f"{OUT_DIR}/cv_folds.csv", index=False)

# 2. Đánh giá baseline và in kết quả mean ± std
models = [
    ("majority", DummyClassifier(strategy="most_frequent")),
    ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
    ("tree3", DecisionTreeClassifier(max_depth=3, random_state=42))
]

results = []
for name, m in models:
    r = cross_validate(m, train[feats], train["y"], cv=cv, groups=train["Entity"],
                       scoring=["accuracy", "f1_macro"])
    acc_mean, acc_std = r['test_accuracy'].mean(), r['test_accuracy'].std()
    f1_mean, f1_std = r['test_f1_macro'].mean(), r['test_f1_macro'].std()
    
    print(f"{name:9s} acc: {acc_mean:.3f} ± {acc_std:.3f} | F1: {f1_mean:.3f} ± {f1_std:.3f}")
    
    results.append({
        "model": name,
        "acc_mean": round(acc_mean, 4), "acc_std": round(acc_std, 4),
        "f1_mean": round(f1_mean, 4), "f1_std": round(f1_std, 4)
    })

pd.DataFrame(results).to_csv(f"{OUT_DIR}/baseline_results.csv", index=False)

majority  acc: 0.265 ± 0.025 | F1: 0.139 ± 0.010
logreg    acc: 0.639 ± 0.085 | F1: 0.629 ± 0.085
tree3     acc: 0.616 ± 0.049 | F1: 0.600 ± 0.063


### 4. Trích xuất Luật Quyết định (Rule Extraction)
In ra cấu trúc phân nhánh của mô hình Cây quyết định. Cấu trúc luật "cứng" dựa trên dữ liệu thực chứng này sẽ được làm tài liệu tham khảo chính xác nhằm nháp ra bộ luật "mờ" (Fuzzy Rules) tĩnh, tránh việc áp đặt tư duy chủ quan sai lệch với thực tế tương quan của y tế cộng đồng.

In [5]:
# Ô 4a: fit cây 1 lần, lưu luật dạng đọc được (4 chữ số)
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(train[feats], train["y"])
txt = export_text(tree, feature_names=feats, decimals=4)
print(txt)
open(f"{OUT_DIR}/tree_rules.txt", "w", encoding="utf-8").write(txt)

|--- schizophrenia <= 0.2545
|   |--- anxiety <= 3.0613
|   |   |--- schizophrenia <= 0.2071
|   |   |   |--- class: 2
|   |   |--- schizophrenia >  0.2071
|   |   |   |--- class: 1
|   |--- anxiety >  3.0613
|   |   |--- eating <= 0.0992
|   |   |   |--- class: 2
|   |   |--- eating >  0.0992
|   |   |   |--- class: 2
|--- schizophrenia >  0.2545
|   |--- bipolar <= 0.3372
|   |   |--- eating <= 0.1756
|   |   |   |--- class: 0
|   |   |--- eating >  0.1756
|   |   |   |--- class: 1
|   |--- bipolar >  0.3372
|   |   |--- anxiety <= 3.7334
|   |   |   |--- class: 0
|   |   |--- anxiety >  3.7334
|   |   |   |--- class: 1



630

In [6]:
# Ô 4b: trích luật cứng tự động (mỗi lá = 1 luật)
import json
def _simplify(tree, n=0, prune=True):
    """Dựng cây rút gọn; gộp hai lá anh em cùng nhãn thành một lá."""
    t = tree.tree_
    if t.children_left[n] == -1:
        return ("leaf", int(tree.classes_[np.argmax(t.value[n][0])]))
    L = _simplify(tree, t.children_left[n], prune)
    R = _simplify(tree, t.children_right[n], prune)
    if prune and L[0] == "leaf" and R[0] == "leaf" and L[1] == R[1]:
        return L
    return ("split", int(t.feature[n]), float(t.threshold[n]), L, R)

def tree_to_rules(tree, feats, prune=True):
    rules = []
    def walk(node, conds):
        if node[0] == "leaf":
            rules.append((conds, node[1])); return
        _, fi, thr, L, R = node
        walk(L, conds + [(feats[fi], "<=", thr)])
        walk(R, conds + [(feats[fi], ">",  thr)])
    walk(_simplify(tree, 0, prune), [])
    return rules

def crisp_predict(rules, X):
    out = np.zeros(len(X), int)
    for conds, cls in rules:
        ok = np.ones(len(X), bool)
        for f, op, thr in conds:
            ok &= (X[f].values <= thr) if op == "<=" else (X[f].values > thr)
        out[ok] = cls
    return out

rules_full = tree_to_rules(tree, feats, prune=False)   # 8 luật, chỉ để đối chiếu
rules = tree_to_rules(tree, feats)                      # bản chính: đã tỉa
print(len(rules_full), "luật đầy đủ ->", len(rules), "luật sau khi tỉa")

json.dump(rules, open(f"{OUT_DIR}/rules_crisp.json", "w"), indent=1)   # hệ mờ đọc file này
pred = crisp_predict(rules, train[feats])
assert (pred == tree.predict(train[feats])).all()
assert (pred == crisp_predict(rules_full, train[feats])).all()          # tỉa không đổi dự đoán

8 luật đầy đủ -> 7 luật sau khi tỉa


In [7]:
# Ô 4c: đo luật cứng theo từng fold (dùng mảng `fold` tạo ở ô baseline)
results = [r for r in results if r["model"] != "crisp_rules"]   # tránh trùng dòng khi chạy lại ô này

accs = [accuracy_score(train["y"][fold==k], pred[fold==k]) for k in range(5)]
f1s  = [f1_score(train["y"][fold==k], pred[fold==k], average="macro") for k in range(5)]
a_m, a_s, f_m, f_s = np.mean(accs), np.std(accs), np.mean(f1s), np.std(f1s)

print(f"{'crisp':9s} acc: {a_m:.3f} ± {a_s:.3f} | F1: {f_m:.3f} ± {f_s:.3f}")
results.append({"model": "crisp_rules",
                "acc_mean": round(a_m,4), "acc_std": round(a_s,4),
                "f1_mean": round(f_m,4),  "f1_std": round(f_s,4)})
pd.DataFrame(results).to_csv(f"{OUT_DIR}/baseline_results.csv", index=False)

crisp     acc: 0.694 ± 0.069 | F1: 0.690 ± 0.072
